# AdamW — Paper-to-Code Mock (Colab)

**Paper:** Decoupled Weight Decay Regularization (AdamW) (Loshchilov & Hutter, 2017) — https://arxiv.org/abs/1711.05101

Medium mock (~30 min). Fill in the `ManualAdam` step (two modes: coupled `L2` and decoupled `AdamW`), run the mechanism demo, then the sanity checks. Reference solution in the last cell.

**The benefit is subtle:** under Adam, L2 regularization (`g += wd*w`) is NOT weight decay, because the penalty gets divided by `sqrt(v_hat)` — so params with large second-moment estimates decay LESS. AdamW decouples decay so every weight shrinks by the same `(1 - lr*wd)`.

In [ ]:
import torch
torch.manual_seed(0)

## 1. Implement the Adam step in two WD modes (YOUR TASK)

- `mode="L2"`: add `wd*p` into the gradient BEFORE the Adam update (coupled / the wrong way).
- `mode="AdamW"`: apply `p *= (1 - lr*wd)` decoupled from the adaptive step.
- Both: standard Adam with `m`, `v`, and bias correction.

In [ ]:
class ManualAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, mode="AdamW"):
        self.params = list(params)
        self.lr, self.eps, self.wd, self.mode = lr, eps, weight_decay, mode
        self.b1, self.b2 = betas
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    @torch.no_grad()
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            g = p.grad
            # TODO (L2 mode): if mode=="L2" and wd!=0 -> g = g + wd * p
            # TODO: update m (EMA of g) and v (EMA of g*g)
            # TODO: bias-correct -> m_hat, v_hat ; compute step = lr*m_hat/(sqrt(v_hat)+eps)
            # TODO (AdamW mode): if mode=="AdamW" and wd!=0 -> p.mul_(1 - lr*wd)
            # TODO: p.sub_(step)
            raise NotImplementedError("fill me in")

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

## 2. Demonstrate the MECHANISM (not a generalization win)
Two params at the same value but with very different second moments `v`. Zero the gradient so only decay moves them, then run one step. Under AdamW both decay by exactly `(1 - lr*wd)`; under Adam+L2 the factors differ (coupled to `1/sqrt(v)`).

In [ ]:
lr, wd = 0.1, 0.1
for mode in ("L2", "AdamW"):
    print(f"\nmode = {mode}")
    for v_state in (1.0, 1e-4):            # param A: big grad history; param B: tiny
        p = torch.tensor([1.0], requires_grad=True)
        opt = ManualAdam([p], lr=lr, weight_decay=wd, mode=mode)
        opt.t = 1000                        # pretend we're deep into training
        opt.v[0].fill_(v_state)             # seed the second moment
        opt.m[0].zero_()                    # no first moment -> isolate decay
        p.grad = torch.zeros_like(p)        # ZERO gradient: only decay can move p
        before = p.item(); opt.step()
        print(f"  v={v_state:>8.0e}  ->  effective decay factor = {p.item()/before:.6f}")
print(f"\nAdamW target (1 - lr*wd) = {1 - lr*wd:.6f} for BOTH params.")

## 3. Sanity checks

In [ ]:
def run(make_ref, make_mine, steps=20, shape=(8,), seed=0):
    torch.manual_seed(seed)
    p0 = torch.randn(shape); grads = [torch.randn(shape) for _ in range(steps)]
    pr = p0.clone().requires_grad_(True); o_r = make_ref([pr])
    pm = p0.clone().requires_grad_(True); o_m = make_mine([pm])
    for g in grads:
        pr.grad = g.clone(); o_r.step(); o_r.zero_grad()
        pm.grad = g.clone(); o_m.step(); o_m.zero_grad()
    return pr.detach(), pm.detach()

# 1) manual Adam (wd=0) matches torch.optim.Adam  [KEY reference check]
r, m = run(lambda ps: torch.optim.Adam(ps, lr=1e-2),
           lambda ps: ManualAdam(ps, lr=1e-2, weight_decay=0.0))
assert torch.allclose(r, m, atol=1e-6)

# 2) manual AdamW matches torch.optim.AdamW  [KEY reference check]
r, m = run(lambda ps: torch.optim.AdamW(ps, lr=1e-2, weight_decay=0.1),
           lambda ps: ManualAdam(ps, lr=1e-2, weight_decay=0.1, mode="AdamW"))
assert torch.allclose(r, m, atol=1e-6)

# 3) bias correction: first step ~= lr
p = torch.tensor([5.0], requires_grad=True)
opt = ManualAdam([p], lr=1e-2, weight_decay=0.0)
p.grad = torch.tensor([0.37]); before = p.item(); opt.step()
assert abs(abs(before - p.item()) - 1e-2) < 1e-3

# 4) AdamW decay is independent of grad AND of v
lr, wd = 0.1, 0.05
p = torch.tensor([3.0], requires_grad=True)
opt = ManualAdam([p], lr=lr, weight_decay=wd, mode="AdamW")
opt.v[0].fill_(123.0); p.grad = torch.zeros_like(p); opt.step()
assert abs(p.item() - 3.0 * (1 - lr * wd)) < 1e-6

# 5) Adam+L2 zero-grad does NOT decay by (1-lr*wd) for nonzero v (coupling)
p = torch.tensor([3.0], requires_grad=True)
opt = ManualAdam([p], lr=lr, weight_decay=wd, mode="L2")
opt.t = 1000; opt.v[0].fill_(1.0); opt.m[0].zero_()
p.grad = torch.zeros_like(p); opt.step()
assert abs(p.item() / 3.0 - (1 - lr * wd)) > 1e-3

# 6) with wd=0, the two modes are identical
torch.manual_seed(7)
p0 = torch.randn(8); gs = [torch.randn(8) for _ in range(10)]
pa = p0.clone().requires_grad_(True); pb = p0.clone().requires_grad_(True)
oa = ManualAdam([pa], lr=1e-2, weight_decay=0.0, mode="L2")
ob = ManualAdam([pb], lr=1e-2, weight_decay=0.0, mode="AdamW")
for g in gs:
    pa.grad = g.clone(); oa.step(); oa.zero_grad()
    pb.grad = g.clone(); ob.step(); ob.zero_grad()
assert torch.allclose(pa.detach(), pb.detach(), atol=1e-7)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class ManualAdamSolution:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, mode="AdamW"):
        self.params = list(params)
        self.lr, self.eps, self.wd, self.mode = lr, eps, weight_decay, mode
        self.b1, self.b2 = betas
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    @torch.no_grad()
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            g = p.grad
            if self.mode == "L2" and self.wd != 0.0:
                g = g + self.wd * p                       # COUPLED: into the grad
            self.m[i].mul_(self.b1).add_(g, alpha=1 - self.b1)
            self.v[i].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            step = self.lr * m_hat / (v_hat.sqrt() + self.eps)
            if self.mode == "AdamW" and self.wd != 0.0:
                p.mul_(1 - self.lr * self.wd)             # DECOUPLED: on the param
            p.sub_(step)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()